# 00 - Quickstart: the whole loop in 5 minutes, offline

**Goal**: see decide -> data -> evaluate -> compare run end to end with no API
key and no GPU, using a deterministic stub model. Once the shape is clear, the
later notebooks swap the stub for a real base model and a real QLoRA fine-tune.

**The task**: turn a free-text support message into a strict JSON `Ticket`.
This is the canonical 'high-frequency narrow task' you migrate off a frontier
API onto a cheap self-hosted small model.

In [ ]:
import sys; sys.path.insert(0, '../src')
from qlora_lab import schema, synth, dataset, predict, evaluate, train, serve, agent

### 1. The contract
Everything is organized around one schema.

In [ ]:
print(schema.SCHEMA_HINT)
t, err = schema.parse_ticket('```json\n{"order_id":"12345","issue":"delivery_delay","sentiment":"negative","priority":"high","summary":"late"}\n```')
print(t, '|', err)

### 2. Generate labeled data (deterministic, no API)

In [ ]:
examples = synth.gen(800, seed=7)
parts = dataset.split(examples, n_test=100, n_val=100)
train, removed = dataset.decontaminate(parts['train'], parts['test'])
print('train', len(train), 'val', len(parts['val']), 'test', len(parts['test']), 'decontam removed', removed)
print(examples[0]['message'])
print(examples[0]['ticket'])

### 3. Score two models on the same test set
We stand in a weak 'base' (fails often) and a strong 'tuned' (rarely fails)
with the offline stub. Note: instantiate each stub **once** and reuse it.

In [ ]:
test = parts['test']
gold = {e['message']: e['ticket'] for e in test}
base_stub = predict.OfflineStub(gold, break_rate=0.45, seed=1)
tuned_stub = predict.OfflineStub(gold, break_rate=0.05, seed=2)
base_preds = [base_stub.chat_completions_create(e['message']) for e in test]
tuned_preds = [tuned_stub.chat_completions_create(e['message']) for e in test]
rb = evaluate.evaluate(base_preds, test, in_price=2.5e-6, out_price=10e-6)
rt = evaluate.evaluate(tuned_preds, test, in_price=0.05e-6, out_price=0.20e-6)
print(evaluate.compare(rb, rt))

The `schema_validity` gap and the cost gap are the two numbers this whole
lab exists to produce for real. Next notebooks replace the stub with actual
models. **The eval set is the game** - everything else serves these numbers.